# Treatment Effects of Different Matching Donation Dosages on Charitable Giving

In this project, we replicated and analyzed several components of the influential study **“Does Price Matter in Charitable Giving? Evidence from a Large-Scale Natural Field Experiment.”** The main objective of the analysis was to examine whether matching donation offers influence donor behavior, specifically whether individuals become more likely to donate and whether they donate larger amounts when a sponsor agrees to match their contributions. The dataset consisted of previous donors who were randomly assigned to different fundraising treatments, allowing the project to be analyzed within a causal inference framework.

We began by cleaning the dataset and removing observations with missing information in key variables. We then examined whether the randomization process created balanced treatment and control groups before treatment assignment. To do this, we conducted difference-in-means t-tests and difference-in-proportions tests on several pre-treatment covariates, including donation history, demographic characteristics, donation frequency, and recency measures. The results showed no statistically significant differences between treatment and control groups, suggesting that the randomization process was successful and reducing concerns about selection bias.

Next, we estimated the Average Treatment Effect (ATE) of the matching offer on two main outcomes: whether an individual donated (out_gavedum) and the amount donated (out_amountgive). We estimated these effects using several statistical approaches, including **simple difference-in-means estimators**, **proportions z-tests**, **t-tests**, and **Ordinary Least Squares (OLS) regressions** with heteroskedasticity-robust standard errors. Across all methods, the matching offer increased both the probability of donating and the average donation amount, and these effects were statistically significant at conventional confidence levels.

To strengthen the robustness of the findings, we also implemented **permutation tests** and **bootstrap methods.** The permutation tests generated empirical null distributions by repeatedly reshuffling treatment assignment under the sharp null hypothesis of no treatment effect. The resulting permutation p-values confirmed that the observed treatment effects were unlikely to have occurred by random chance. In addition, bootstrap resampling methods were used to estimate confidence intervals for the treatment effects without relying heavily on normality assumptions. The bootstrap confidence intervals remained entirely above zero, further supporting the statistical significance of the matching campaign.

The project then moved beyond average effects and explored heterogeneous treatment effects across different experimental conditions. We estimated separate regressions to study whether different matching ratios (1:1, 2:1, and 3:1), different maximum matching thresholds ($25k, $50k, and $100k), and different suggested ask amounts affected donor behavior differently. While some coefficients were directionally positive, none of these treatment “dose” variations produced statistically significant differences in donation probability relative to their base groups. This suggests that simply offering a match mattered more than the exact size of the match or the suggested donation amount.

We also incorporated donor covariates into the regression models, including gender, couple status, previous donation behavior, donation frequency, recency, and years since first donation. Including these controls did not materially change the estimated treatment effect, reinforcing the credibility of the randomized design. However, several donor history variables, particularly donation frequency and recency, were significantly related to donation behavior, indicating that prior engagement strongly predicts future charitable giving. A correlation matrix analysis further confirmed these relationships and showed consistency between the correlation structure and the regression findings.

Finally, we analyzed heterogeneous treatment effects across political environments by comparing donors in red states and blue states. Separate regressions showed that the matching campaign had a strong and statistically significant positive effect in red states, increasing donation probability by about one percentage point, while the treatment effect in blue states was small and statistically insignificant. We then estimated an interaction regression on the full dataset to formally test whether the treatment effects differed between political groups. The interaction term between treatment assignment and red-state status was positive and statistically significant, confirming that the matching campaign was significantly more effective in red states than in blue states.

In conclusion, this project demonstrated how randomized experiments and causal inference methods can be used to estimate the effectiveness of fundraising strategies. By combining classical hypothesis testing, regression analysis, permutation inference, bootstrap methods, and heterogeneous treatment effect analysis, the study provided a comprehensive examination of how matching donation campaigns influence charitable giving behavior.

In [2]:
import pandas as pd
import numpy as np
import io
import requests
from scipy.stats import ttest_ind
from statsmodels.stats.proportion import proportions_ztest
import statsmodels.formula.api as smf

In [3]:
df=pd.read_csv('charitable_data.csv') 
df

,treatment,treat_ratio2,treat_ratio3,treat_size25,treat_size50,treat_size100,treat_sizeno,treat_askd1,treat_askd2,treat_askd3,...,red0_missing,blue0_missing,redcty_missing,bluecty_missing,pwhite_missing,pblack_missing,page18_39_missing,median_hhincome_missing,psch_atlstba_missing,pop_propurban_missing
0,1,1,0,0,0,0,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,1,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,1,0,0,1,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50078,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
50079,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
50080,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
50081,1,0,0,1,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


In [4]:
# removing missing values 
df['any_missing'] = df[df.columns[-19:]].max(axis=1)
df = df[df.any_missing==0]
df

,treatment,treat_ratio2,treat_ratio3,treat_size25,treat_size50,treat_size100,treat_sizeno,treat_askd1,treat_askd2,treat_askd3,...,blue0_missing,redcty_missing,bluecty_missing,pwhite_missing,pblack_missing,page18_39_missing,median_hhincome_missing,psch_atlstba_missing,pop_propurban_missing,any_missing
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,1,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,1,0,0,1,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
5,1,0,1,0,0,1,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50078,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
50079,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
50080,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
50081,1,0,0,1,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


# In the following code :

For each of the following four variables we will calculate the difference in means between treatment (matching offer) 
and control (no matching offer) using the Simple Difference in Outcomes, the t-statistic, and the p-value.

- 1-Number of months since last donation
- 2-Highest previous contribution
- 3-Number of prior donations
- 4-Number of years since initial donation

to see if there is any evidence of an imbalance in pre-treatment covariates. 

In [5]:
# variables to test
variables = ['mrm2', 'hpa', 'freq', 'years']

for var in variables:
    
    # treatment group
    treated = df[df['treatment'] == 1][var]
    
    # control group
    control = df[df['treatment'] == 0][var]
    
    # difference in means
    diff_means = treated.mean() - control.mean()
    
    # t-test
    t_stat, p_value = ttest_ind(treated, control)
    
    print(f"\nVariable: {var}")
    print(f"Difference in Means: {diff_means:.4f}")
    print(f"T-statistic: {t_stat:.4f}")
    print(f"P-value: {p_value:.4f}")


Variable: mrm2
Difference in Means: 0.0074
T-statistic: 0.0622
P-value: 0.9504

Variable: hpa
Difference in Means: 0.7014
T-statistic: 1.0173
P-value: 0.3090

Variable: freq
Difference in Means: -0.0037
T-statistic: -0.0328
P-value: 0.9738

Variable: years
Difference in Means: -0.0563
T-statistic: -1.0360
P-value: 0.3002


These varables are called pre-treatment covariates because they already existed before the experiment/treatment happened.
Like months since last donation, highest previous donation, years as donor. All happened before the charity sent the matching offer.

The treatment was **receiving the matching donation letter** So these variables cannot be caused by treatment.

That is important because in randomized experiments we want **treatment and control groups to look similar BEFORE treatment starts.** If these pre-treatment variables are very different between groups, randomization may have failed or created imbalance by chance.

# Test Interpertation

The null hypothesis for each variable is:

H0: Treatment mean − Control mean = 0

Meaning: **There is no difference between treatment and control groups for that pre-treatment variable.**

The alternative hypothesis is:

H1: Treatment mean − Control mean ≠ 0

Meaning: **The two groups are different.**

Now important point:

Even if the sample difference is not exactly zero (which almost always happens in real data), we do not immediately conclude imbalance.

We ask **Is the observed difference large enough relative to random variation?**

That is what the t-test and p-value measure. If: p < 0.05 then we reject the null and conclude:

There is statistical evidence that treatment and control groups differ for that variable. This could indicate imbalance.

If: p > 0.05 then we fail to reject the null. Meaning We do not have evidence that the groups are different.

That is good in randomized experiments because it suggests treatment assignment was balanced.

This relates directly to **causal inference and selection bias.**

If pre-treatment covariates are balanced, then:

- treatment and control groups look similar before treatment
- treatment assignment appears random
- selection bias is reduced

So differences in outcomes after treatment are more likely to be caused by the treatment itself rather than pre-existing differences between groups.


But here all p-values are much larger than 0.05.
So we fail to reject: **mean_treatment = mean_control**  for all variables.

Also the differences in means are very small for :

- mrm2: 0.0074 months
- freq: -0.0037 donations
- years: -0.0563 years

Even hpa difference is small relative to donation sizes. So there is no strong evidence of imbalance in pre-treatment covariates. This is what we hope to see in a randomized experiment because it suggests the treatment and control groups were comparable before the matching offer was sent.

# For the following code: 

We will use the difference in proportions t-test to calculate the difference in means between treatment (matching offer) and control (no matching offer), the t-statistic, and the p-value, for the following 3 binary variables

- Already donated in 2005
- Female
- Couple

and evaluate if we see any evidence of an imbalance in pre-treatment covariates. 

In [6]:
# binary variables
variables = ['year5', 'female', 'couple']

for var in variables:
    
    # treatment group
    treated = df[df['treatment'] == 1][var]
    
    # control group
    control = df[df['treatment'] == 0][var]
    
    # difference in proportions (means)
    diff_means = treated.mean() - control.mean()
    
   
    t_stat, p_value = ttest_ind(treated, control)
    
    print(f"\nVariable: {var}")
    print(f"Difference in Proportions: {diff_means:.4f}")
    print(f"T-statistic: {t_stat:.4f}")
    print(f"P-value: {p_value:.4f}")


Variable: year5
Difference in Proportions: -0.0077
T-statistic: -1.5757
P-value: 0.1151

Variable: female
Difference in Proportions: -0.0066
T-statistic: -1.5050
P-value: 0.1323

Variable: couple
Difference in Proportions: 0.0001
T-statistic: 0.0274
P-value: 0.9781


# Interperatation of the tets 

These results again suggest there is no strong evidence of imbalance between treatment and control groups. So we fail to reject the null hypothesis for all three variables. This means: 


- the proportion of donors who donated in 2005 is similar across groups

- the proportion of females is similar across groups

- the proportion of couples is similar across groups


Also the differences in proportions are very small:

- year5: -0.0077

- female: -0.0066

- couple: 0.0001


So there is no evidence of imbalance in these pre-treatment binary covariates. This supports the idea that the random assignment worked well and reduces concerns about selection bias in the causal analysis.

# For the following code


We will concentrate on estimating treatment effects. In this section, we will calculate the treatment effects and
statistical significance in 3 different ways. Focus on the two outcome variables:

- did the member give anything in response to this campaign(Treatment)
- how much did the donor give in response to this campaign?

Then we will estimatie treatment effects with t-test.

We will Use the simple difference in outcomes to estimate the average treatment effect for both outcome variables.

Then we use the proportions z test and the difference in means t-test to calculate test statistics and p-values. 

In [7]:

# 1. Outcome: gave anything

treated_give = df[df['treatment'] == 1]['out_gavedum']
control_give = df[df['treatment'] == 0]['out_gavedum']

ate_give = treated_give.mean() - control_give.mean()

# z-test for difference in proportions
successes = np.array([treated_give.sum(), control_give.sum()])
nobs = np.array([treated_give.shape[0], control_give.shape[0]])

z_stat, p_value_give = proportions_ztest(successes, nobs)

print("Outcome: out_gavedum")
print(f"Treatment effect: {ate_give:.4f}")
print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value_give:.4f}")


# 2. Outcome: amount donated
treated_amount = df[df['treatment'] == 1]['out_amountgive']
control_amount = df[df['treatment'] == 0]['out_amountgive']

ate_amount = treated_amount.mean() - control_amount.mean()

# t-test for difference in means
t_stat, p_value_amount = ttest_ind(treated_amount, control_amount)

print("\nOutcome: out_amountgive")
print(f"Treatment effect: {ate_amount:.4f}")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value_amount:.4f}")

Outcome: out_gavedum
Treatment effect: 0.0049
Z-statistic: 3.4517
P-value: 0.0006

Outcome: out_amountgive
Treatment effect: 0.1829
T-statistic: 2.1168
P-value: 0.0343


In above code we have two dependent varaibales:
    
- 1- out_gavedum: a binary variable and an outcome/dependent variable

1 = donor gave something

0 = donor gave nothing

and we are trying to see whether treatment changes this outcome.


- 2- out_amountgive

It is a continuous quantitative outcome variable. it measures How much money the donor gave.

It is also a dependent/outcome variable because treatment may affect it.




### Then we choose tests based on the type of outcome variable

| Variable Type | Example | Test |
|---|---|---|
| Binary (0/1) | donated or not | z-test for proportions |
| Continuous numeric | donation amount | t-test for means |

---

### 1. Difference in Proportions Z-Test

Used for binary outcomes.Here:

$$
\text{out_gavedum}
$$

The treatment effect is:

$$
\hat p_T - \hat p_C
$$

where:

- \( hat p_T \) = proportion donating in treatment group  
- \( hat p_C \) = proportion donating in control group  

---

# Z-statistic

$$
z=\frac{\hat p_T-\hat p_C}{SE(\hat p_T-\hat p_C)}
$$

---

# Standard error

$$
SE=\sqrt{\hat p(1-\hat p)\left(\frac{1}{n_T}+\frac{1}{n_C}\right)}
$$

where:

- \( \hat p \) = pooled proportion  
- \( n_T \) = treatment sample size  
- \( n_C \) = control sample size  

---

# Pooled proportion

$$
\hat p=\frac{x_T+x_C}{n_T+n_C}
$$

where:

- \( x_T \) = number of successes in treatment  
- \( x_C \) = number of successes in control  

---

# Interpertation of the equation :

The numerator:

$$
\hat p_T-\hat p_C
$$

is the observed treatment effect.

The denominator:

$$
SE
$$

measures:**how much random variation we expect just from sampling noise.**

So the z-statistic asks: **Is the observed difference large relative to random variation?**

Large absolute z-statistic:

$$
|z| \text{ large}
$$

means: evidence against the null hypothesis.

---

#### we use z-test here because 1- outcome is binary and 2-with large samples, proportions become approximately normally distributed (Central Limit Theorem)  
So we can use the normal (z) distribution.

---

### 2. Difference in Means T-Test

This test is used for continuous variables.Here:

$$
\text{out\_amountgive}
$$

---

# Treatment effect

$$
\bar Y_T-\bar Y_C
$$

where:

- \( \bar Y_T \) = average donation in treatment  
- \( \bar Y_C \) = average donation in control  

---

# T-statistic

$$
t=\frac{\bar Y_T-\bar Y_C}{SE(\bar Y_T-\bar Y_C)}
$$

---

# Standard error

$$
SE=\sqrt{\frac{s_T^2}{n_T}+\frac{s_C^2}{n_C}}
$$

where:

- \( s_T^2 \) = variance of treatment group  
- \( s_C^2 \) = variance of control group  
- \( n_T,n_C \) = sample sizes  

---

**Variance** measures spread/noise in data. Higher variance: 1- more uncertainty   and 2 - larger SE  

---

### we use t-test because:

- outcome is continuous  
- population variance is unknown  
- we estimate variance from the sample  

That is why we use the t-distribution instead of the normal distribution.

## Test result 

Both treatment effects are statistically significant at the 95% confidence level

---

### 1. Probability of donating

Treatment effect:

$$
0.0049
$$

This means the matching offer increased the probability of donating by about:

$$
0.49 \text{ percentage points}
$$

Since:

$$
p\text{-value} = 0.0006 < 0.05
$$

we reject the null hypothesis.

So the matching offer had a statistically significant positive effect on whether people donated.

---

### 2. Donation amount

Treatment effect:

$$
0.1829
$$

This means the matching offer increased the average donation amount by about:

$$
\$0.18 \text{ per person}
$$

Since:

$$
p\text{-value} = 0.0343 < 0.05
$$

we reject the null hypothesis.

So the matching offer also had a statistically significant positive effect on donation amount.

As a final result we can say Matching offers increased 1- the probability of donating and 2- the average amount donated

# For the following code 

We will stimating treatment effects with linear regressions. 

In [8]:

# 1. Outcome: donated or not

mod1 = smf.ols(
    formula='out_gavedum ~ treatment',
    data=df
)

res1 = mod1.fit(cov_type='HC3')

print(res1.summary())

# 95% confidence interval
print("\n95% CI for out_gavedum:")
print(res1.conf_int())

# 2. Outcome: donation amount

mod2 = smf.ols(
    formula='out_amountgive ~ treatment',
    data=df
)

res2 = mod2.fit(cov_type='HC3')

print(res2.summary())

print("\n95% CI for out_amountgive:")
print(res2.conf_int())

                            OLS Regression Results                            
Dep. Variable:            out_gavedum   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     12.90
Date:                Wed, 13 May 2026   Prob (F-statistic):           0.000329
Time:                        14:02:56   Log-Likelihood:                 24504.
No. Observations:               46513   AIC:                        -4.900e+04
Df Residuals:                   46511   BIC:                        -4.899e+04
Df Model:                           1                                         
Covariance Type:                  HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.0176      0.001     16.669      0.0

# This regression estimates

$$
Y_i = \beta_0 + \beta_1 \text{Treatment}_i + \varepsilon_i
$$

where:

- ( Y_i ) = outcome  
- (Treatment) = 1 if matching offer received  
- ( beta_1 ) = Average Treatment Effect (ATE)  

The important coefficient is:

$$
\text{treatment}
$$

The confidence interval tells us the range of plausible values for the treatment effect.

If the 95% confidence interval crosses 0:

$$
\text{lower} < 0 < \text{upper}
$$

then not statistically significant.

If the confidence interval does NOT cross 0:

then: statistically significant at the 95% confidence level.

---

# First regression

$$
\text{probability of donating} \sim \text{treatment}
$$

Means dependent/outcome variable: probability of donating  

independent variable is treatment  

So we are estimating: **Does receiving the matching offer change the probability of donating?**

---

# Second regression

$$
\text{donation amount} \sim \text{treatment}
$$

means dependent/outcome variable is donation amount and independent variable is treatment  

So we are estimating **Does receiving the matching offer change the donation amount**

---

# Confidence interval interpertaion :

To better understanf CI suppose regression estimates treatment effect:

$$
0.18
$$

This is our best estimate from the sample. But samples contain randomness/noise.So the true effect in the population may not be exactly:

$$
0.18
$$

The confidence interval gives a reasonable range for the true effect. Example:

$$
95\% \text{ CI} = [0.03,\ 0.34]
$$

Meaning:**Based on our data, the true treatment effect is likely between 0.03 and 0.34**


### the point is crossing 0 matter because for example 


$$
[-0.05,\ 0.30]
$$

contains zero.

So maybe:

- true effect is positive  
- OR true effect is negative  
- OR maybe no effect at all  

So we cannot confidently say treatment had an effect.

But:

$$
[0.03,\ 0.34]
$$

does NOT contain:

$$
0
$$

All plausible values are positive. So we conclude treatment likely has a positive effect.

---

# Difference between regression vs t-test / z-test

In this simple experiment like we have with one independent variable **regression and t/z tests are basically estimating the SAME treatment effect.**

The regression coefficient on treatment:

$$
\beta_1
$$

is mathematically equal to:

$$
\text{treated mean} - \text{control mean}
$$

So both methods answer:**Is treatment different from control?**

---

# Main differences

| T/Z test | Regression |
|---|---|
| Simple comparison | More flexible |
| Usually one variable at a time | Can add many control variables |
| Harder to extend | Easy to extend |
| Gives test statistic/p-value | Gives coefficients, SE, CI, predictions |

---

# As a short note regression is powerful: 

Beacuse we can later add more variables and it has flexibility


$$
\text{donation amount} \sim \text{treatment} + \text{female} + \text{years} + \text{frequency}
$$

Now regression can:

- control for covariates  
- estimate adjusted effects  
- include interactions  
- handle many variables together  

T-tests cannot easily do that. 

# Result of the regression test :

### 1. Regression for probability of donating

Dependent variable:

$$
\text{probability of donating}
$$

Meaning: **Did the donor donate or not?**

---

#### Regression equation

$$
\text{probability of donating}_i
=
\beta_0
+
\beta_1 \text{treatment}_i
+
\varepsilon_i
$$

---

#### Intercept

Intercept:

$$
0.0176
$$

This is the average donation probability for the control group.

Meaning **About 1.76% of people in the control group donated**

---

#### Treatment coefficient

$$
\text{treatment} = 0.0049
$$

This is the Average Treatment Effect (ATE).

Meaning **Receiving a matching offer increased donation probability by about 0.49 percentage points.**

---

#### Statistical significance

$$
p\text{-value} = 0.000
$$

Very small.

So we reject:

$$
H_0 : \beta_1 = 0
$$

and conclude treatment had a statistically significant positive effect.

---

#### Confidence interval

$$
[0.0022,\ 0.0075]
$$

This interval does NOT cross:

$$
0
$$

So all plausible effects are positive. That confirms statistical significance.

---

### 2. Regression for donation amount

Dependent variable:

$$
\text{donation amount}
$$

Meaning **How much money was donated?**

---

#### Regression equation

$$
\text{donation amount}_i
=
\beta_0
+
\beta_1 \text{treatment}_i
+
\varepsilon_i
$$

---

#### Intercept

Intercept:

$$
0.8048
$$

Average donation amount in the control group.

Meaning **Control group donated about \$0.80 on average.**

---

#### Treatment coefficient

$$
\text{treatment} = 0.1829
$$

Meaning **Matching offer increased average donation amount by about \$0.18 per person.**

---

#### Statistical significance

$$
p\text{-value} = 0.028
$$

Since:

$$
0.028 < 0.05
$$

the effect is statistically significant at the 95% confidence level.

---

#### Confidence interval

$$
[0.0198,\ 0.3460]
$$

Again, the interval does NOT cross:

$$
0
$$

So treatment effect is statistically significant.

---


Both regressions show:

- positive treatment effects  
- statistically significant effects  
- confidence intervals entirely above zero  

So the matching offer increased:

- probability of donating  
- average donation amount  

---

#### AS a quick short note :


The treatment coefficients:

$$
0.0049
$$

and

$$
0.1829
$$

are exactly the same as our earlier simple differences in means.That happens because with a single binary treatment variable:

$$
\text{OLS coefficient} = \text{treated mean} - \text{control mean}
$$

So regression here is reproducing the same ATE mathematically.

# For the follwoing code 

We will estimating treatment effects with permutation tests.

we will create a “shuffle_treatment” to assign treatment under the sharp null that randomly assigns treatment assuming 
exchangeability and then shuffle treatment 1000 times.

We also know rhat the 33.3% of samples are in control and 66.7 are in treatment.
   


In [11]:
dff = df.copy()

np.random.seed(123)

outcomes = ['out_gavedum', 'out_amountgive']

# number of permutations
num_permutations = 1000

for outcome in outcomes:
    
    # observed ATE using real treatment
    observed_ate = (
        dff[dff['treatment'] == 1][outcome].mean()
        - dff[dff['treatment'] == 0][outcome].mean()
    )
    
    # storing shuffled ATEs
    shuffled_ates = []
    
    for i in range(num_permutations):
        
        # randomly assigning treatment under sharp null
        # 66.7% treatment, 33.3% control
        dff['shuffle_treatment'] = np.random.choice(
            [0, 1],
            size=len(dff),
            p=[0.333, 0.667]
        )
        
        # calculating fake ATE using shuffled treatment
        shuffled_ate = (
            dff[dff['shuffle_treatment'] == 1][outcome].mean()
            - dff[dff['shuffle_treatment'] == 0][outcome].mean()
        )
        
        shuffled_ates.append(shuffled_ate)
    
    # converting list to array
    shuffled_ates = np.array(shuffled_ates)
    
    # two-sided p-value
    p_value = np.mean(np.abs(shuffled_ates) >= abs(observed_ate))
    
    print(f"\nOutcome: {outcome}")
    print(f"Observed ATE: {observed_ate:.4f}")
    print(f"Permutation p-value: {p_value:.4f}")


Outcome: out_gavedum
Observed ATE: 0.0049
Permutation p-value: 0.0000

Outcome: out_amountgive
Observed ATE: 0.1829
Permutation p-value: 0.0350




We tried permutation test here because permutation tests are more **data-driven** and **require fewer theoretical assumptions**  


Z-tests and t-tests rely on mathematical assumptions like:

- normal approximation  
- t-distribution  
- variance formulas  
- large sample theory  

They use theoretical distributions like **“What would happen under repeated sampling according to statistical theory?”**

Permutation tests do not rely heavily on distribution assumptions. Instead they ask:

**“What kinds of treatment effects would appear purely by random assignment?”**

So we create the null distribution directly from the data by reshuffling treatment many times.  


It is saying if treatment truly does nothing **random reshuffling should create effects similar to the observed one**

If the observed effect is much larger than shuffled effects it is **evidence of a real treatment effect.**

Permutation tests are especially useful when:

- A. Small samples: Because: normal/t approximations may fail  

- B. Data are skewed or weird like our data with many zeros and few very large donations which is NOT very normal.

Permutation tests handle this better.

- C. We want fewer assumptions: Permutation tests mainly rely on: exchangeability and random assignment instead of strong distribution assumptions.

- D. Randomized experiments : Permutation tests fit very naturally with **randomized controlled trials** Because treatment assignment itself was random.


#### Big picture

| Method | Main idea |
|---|---|
| z-test | uses normal distribution theory |
| t-test | uses t-distribution theory |
| regression | estimates effect + uses t/z tests internally |
| permutation test | creates empirical null distribution by reshuffling data |



#### we use 33% control and 67% treatment instead of 50/50 because the original experiment itself was not designed as a:

$$
50/50
$$

experiment.

The real data apparently assigned approximately:

- \(33.3\%\) → control  
- \(66.7\%\) → treatment  

So in permutation testing we try to mimic the original experimental design.That is why during reshuffling we preserve roughly the same treatment probability:

$$
p=[0.333,\ 0.667]
$$

If the real experiment had been:

$$
50/50
$$

then we would use:

$$
p=[0.5,\ 0.5]
$$


## Result interpertation for permutation Test

The permutation tests support the same conclusion as the earlier z-tests, t-tests, and regressions.

For out_gavedum, the observed Average Treatment Effect (ATE) was 0.0049, meaning the matching offer increased the 
probability of donating by about 0.49 percentage points. The permutation p-value was essentially 0.0000, which means that after randomly reshuffling treatment assignment 1000 times under the sharp null hypothesis of no treatment effect, almost none of the fake shuffled treatment effects were as large as the real observed effect. This provides very strong evidence that the increase in donation probability was not caused by random chance.

For out_amountgive, the observed ATE was 0.1829, meaning the matching offer increased the average donation amount by about $0.18 per donor. The permutation p-value was 0.0350, which is below the standard 0.05 significance threshold. This indicates that only about 3.5% of the shuffled treatment assignments produced effects as extreme as the observed one, providing evidence that the increase in donation amount was also statistically significant.

Overall, the permutation tests confirm the earlier findings: the matching offer had a positive and statistically significant effect on both the likelihood of donating and the amount donated.

# for following code 

Now for the next step we will conduct other method named Boostraping. 

In [13]:
np.random.seed(123)

outcomes = ['out_gavedum', 'out_amountgive']

# number of bootstrap samples
B = 1000

for outcome in outcomes:
    
  
    observed_ate = (
        df[df['treatment'] == 1][outcome].mean()
        - df[df['treatment'] == 0][outcome].mean()
    )
    
    # storing bootstrap ATEs
    bootstrap_ates = []
    
    for i in range(B):
        
        # bootstraping sample with replacement
        boot_df = df.sample(n=len(df), replace=True)
        
        # calculating bootstrap ATE
        boot_ate = (
            boot_df[boot_df['treatment'] == 1][outcome].mean()
            - boot_df[boot_df['treatment'] == 0][outcome].mean()
        )
        
        bootstrap_ates.append(boot_ate)
    
    # confidence interval
    lower = np.percentile(bootstrap_ates, 2.5)
    upper = np.percentile(bootstrap_ates, 97.5)
    
    print(f"\nOutcome: {outcome}")
    print(f"Observed ATE: {observed_ate:.4f}")
    print(f"95% Bootstrap CI: [{lower:.4f}, {upper:.4f}]")


Outcome: out_gavedum
Observed ATE: 0.0049
95% Bootstrap CI: [0.0021, 0.0074]

Outcome: out_amountgive
Observed ATE: 0.1829
95% Bootstrap CI: [0.0249, 0.3331]


## Boostrapping :

Now Instead of reshuffling treatment labels like permutation tests, we do bootstraping. we go the following steps :

- Randomly sample rows from the dataset
- Allow rows to be selected multiple times (“with replacement”)
- Create a new fake dataset of the same size
- Compute the treatment effect
- Repeat this many times (like 1000 times)

So we get: ATE1, ATE2, ATE3, ..., ATE1000 Then we study the distribution of these ATEs.
    
    
## Boostraping Result Interpertaion :


The bootstrap results support the same conclusion as the t-tests, regressions, and permutation tests.
For out_gavedum, the observed Average Treatment Effect (ATE) was 0.0049, meaning the matching offer increased the probability of donating by about 0.49 percentage points. The 95% bootstrap confidence interval was approximately [0.0021, 0.0074]. Since this interval does not include 0, the treatment effect is statistically significant. This suggests that the true increase in donation probability is likely between about 0.21 and 0.74 percentage points.
For out_amountgive, the observed ATE was 0.1829, meaning the matching offer increased the average donation amount by about $0.18 per donor.

The 95% bootstrap confidence interval was [0.0249, 0.3331]. Again, because the interval does not cross 0, the effect is statistically significant at the 95% confidence level. This suggests the true increase in donation amount is likely between about $0.02 and $0.33 per donor.
An important difference from the permutation test is that the bootstrap distribution was centered around the observed treatment effects (0.0049 and 0.1829) rather than around zero. This is because bootstrap is estimating the uncertainty of the observed effect, while permutation testing assumes the null hypothesis of no treatment effect.
Overall, the bootstrap analysis confirms that the matching offer had a positive and statistically significant effect on both the probability of donating and the amount donated.

# For the following code 


We are now going to examine the different cells and heterogeneous treatment effects. 
    
- 1-We will use a linear regression to estimate the treatment effects of differing matching offers, a 1:1 ratio, 2:1 
ratio, and 3:1 ratio to see if there is any evidence to suggest the different doses have an impact.
    
- 2- We will Use a linear regression to estimate the treatment of different maximum donation thresholds to if there is
any evidence to suggest the different dosages have an impact.  

- 3- We will use a linear regression to estimate the treatment of the ask amount to see if there is any evidence to
suggest the different dosages have an impact. 

In [14]:

# We focus only on treated donors for dose comparisons because this dosage(like treat_ratio2 + treat_ratio3 ) is just for treatment groups. 
df_treat = df[df['treatment'] == 1].copy()

# we can use the whole data but we need to include tretament variable in the regression. 

In [17]:
# 1. Matching ratio: 1:1, 2:1, 3:1
# Here, the omitted/base group is the 1:1 matching ratio.

mod_ratio = smf.ols(
    formula='out_gavedum ~ treat_ratio2 + treat_ratio3',
    data=df_treat
)

res_ratio = mod_ratio.fit(cov_type='HC3')

print(res_ratio.summary())

                            OLS Regression Results                            
Dep. Variable:            out_gavedum   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                     1.005
Date:                Thu, 14 May 2026   Prob (F-statistic):              0.366
Time:                        12:44:25   Log-Likelihood:                 15206.
No. Observations:               31018   AIC:                        -3.041e+04
Df Residuals:                   31015   BIC:                        -3.038e+04
Df Model:                           2                                         
Covariance Type:                  HC3                                         
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept        0.0208      0.001     14.851   

## Matching ratio interpretation

Each treated donor belongs to only one matching-ratio condition:

- 1:1 match  
- 2:1 match  
- 3:1 match  

This means a donor cannot simultaneously receive both:

- 2:1  
- and 3:1  

offers.

That is why: treatment ratio 2 and reatment ratio 3 are dummy variables comparing each condition to the omitted/base category.

---

### Omitted/base category

The omitted/base category here is **1:1 matching ratio**

because it is not explicitly included in the regression.

---

### the regression table

Regression:

$$
\text{probability of donating}
\sim
\text{2:1 match dummy}
+
\text{3:1 match dummy}
$$

Outcome:

$$
\text{probability of donating}
$$

which means:

- \(1 =\) donor gave  
- \(0 =\) donor did not give  

So coefficients represent: **changes in donation probability.**

---

### Intercept

Intercept:

$$
0.0208
$$

This is the average donation probability for the omitted/base group.

Base group: **1:1 matching ratio**

So donors receiving a **1:1 match** donated at about:

$$
2.08\%
$$

---

### 2:1 matching ratio coefficient

Coefficient:

$$
0.0022
$$

Meaning **1 matching increased donation probability by about 0.22 percentage points relative to 1:1 matching.**

So approximate donation probability becomes:

$$
2.08\% + 0.22\% \approx 2.30\%
$$

But:

$$
p\text{-value} = 0.285
$$

which is much larger than:

$$
0.05
$$

So this difference is NOT statistically significant.

Meaning **We do not have strong evidence that 2:1 performs differently from 1:1.**

---

### 3:1 matching ratio coefficient

Coefficient:

$$
0.0027
$$

Meaning **3:1 matching increased donation probability by about 0.27 percentage points relative to 1:1.**

Approximate donation probability:

$$
2.08\% + 0.27\% \approx 2.35\%
$$

But:

$$
p\text{-value} = 0.185
$$

Again larger than:

$$
0.05
$$

So this difference is also NOT statistically significant.

---

### Confidence intervals

For both coefficients **confidence intervals cross 0.**

Example:

$$
[-0.001,\ 0.007]
$$

Which means true effect could be positive , could be zero , could even be slightly negative so evidence is weak.

---

So we can say the estimated effects for 2:1 matches and 3:1 matches are slightly positive relative to the 1:1 match but none of the differences are statistically significant.

Therefore there is no strong evidence that larger matching ratios generated meaningfully different donation probabilities among treated donors.

In [18]:
# 2. Maximum donation threshold: no max, $25k, $50k, $100k
# Here, the omitted/base group is the treatment group with no maximum threshold.

mod_size = smf.ols(
    formula='out_gavedum ~ treat_size25 + treat_size50 + treat_size100',
    data=df_treat
)

res_size = mod_size.fit(cov_type='HC3')

print(res_size.summary())

                            OLS Regression Results                            
Dep. Variable:            out_gavedum   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                    0.1045
Date:                Thu, 14 May 2026   Prob (F-statistic):              0.957
Time:                        13:15:15   Log-Likelihood:                 15205.
No. Observations:               31018   AIC:                        -3.040e+04
Df Residuals:                   31014   BIC:                        -3.037e+04
Df Model:                           3                                         
Covariance Type:                  HC3                                         
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept         0.0230      0.002     13.534

# Maximum Threshold 

### Independent variables

The independent variables here are:

- 1. \$25,000 threshold : Donor received a matching campaign with a maximum matching limit of:

$$
\$25{,}000
$$



- 2. \$50,000 threshold: Donor received a matching campaign with a maximum matching limit of:

$$
\$50{,}000
$$

---

- 3. \$100,000 threshold: Donor received a matching campaign with a maximum matching limit of:

$$
\$100{,}000
$$

---

### Omitted/base group

The omitted/base group is: **treated donors with NO maximum matching threshold.**

So all coefficients compare each threshold condition to the: **“no maximum limit” condition.**

---

## Interpreting the regression

Outcome variable:

$$
\text{probability of donating}
$$

Meaning:

- \(1 =\) donor gave  
- \(0 =\) donor did not give  

So coefficients represent:**changes in donation probability.**

---

### Intercept

Intercept:

$$
0.0230
$$

This is the average donation probability for the omitted/base group:**treated donors with no maximum threshold.**

So donors in the no-limit group donated at about:

$$
2.30\%
$$

---

### 25k threshold coefficient

Coefficient:

$$
-0.0012
$$

Meaning that $25k maximum threshold reduced donation probability by about 0.12 percentage points relative to having no maximum threshold.

Approximate donation probability:

$$
2.30\% - 0.12\% \approx 2.18\%
$$

But:

$$
p\text{-value} = 0.598
$$

which is much larger than:

$$
0.05
$$

So this effect is NOT statistically significant.


### 50k threshold coefficient

Coefficient:

$$
-0.0002
$$

Meaning: **A \$50k threshold reduced donation probability by about 0.02 percentage points relative to no threshold.**

Very tiny effect.

And:

$$
p\text{-value} = 0.918
$$

Not statistically significant.

---

### $100k threshold coefficient

Coefficient:

$$
-0.0005
$$

Meaning: **A \$100k threshold reduced donation probability by about 0.05 percentage points relative to no threshold.**

Again extremely small.

And:

$$
p\text{-value} = 0.838
$$

Not statistically significant.

---

### Confidence intervals

All confidence intervals cross:

$$
0
$$

Which means true effects could be positive or negative or zero . So evidence is weak.

---


So there is no strong evidence that different maximum donation thresholds affected donation probability.

In [19]:
# 3. Ask amount: highest previous, 1.25x, 1.5x
# Here, the omitted/base group is treat_askd1 meaning the ask amount equal to the highest previous contribution.
mod_ask = smf.ols(
    formula='out_gavedum ~ treat_askd2 + treat_askd3',
    data=df_treat
)

res_ask = mod_ask.fit(cov_type='HC3')

print(res_ask.summary())

                            OLS Regression Results                            
Dep. Variable:            out_gavedum   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                    0.1132
Date:                Thu, 14 May 2026   Prob (F-statistic):              0.893
Time:                        13:35:37   Log-Likelihood:                 15205.
No. Observations:               31018   AIC:                        -3.040e+04
Df Residuals:                   31015   BIC:                        -3.038e+04
Df Model:                           2                                         
Covariance Type:                  HC3                                         
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept       0.0221      0.001     15.302      

# Suggested donation amount variables

These variables are about the  **“suggested donation amount”** like written in the fundraising letter.

The charity suggested how much the donor should give based on **their previous donation history.**

---


### 1. Base group

Suggested donation amount was **equal to donor's highest previous contribution.**

Example If donor previously gave:

$$
\$100
$$

then suggested amount was:

$$
\$100
$$

This is the omitted/base group in the regression.

---

### 2. 1.25× previous donation

Suggested amount was:

$$
1.25 \times \text{highest previous donation}
$$

Example Previous highest donation:

$$
\$100
$$

Suggested amount becomes:

$$
\$125
$$

---

### 3. 1.5× previous donation

Suggested amount was:

$$
1.5 \times \text{highest previous donation}
$$

for exampl previous highest donation:

$$
\$100
$$

Suggested amount becomes:

$$
\$150
$$

---


In this part the researchers want to know:**Does asking people for larger suggested amounts change whether they donate?**

---

### Interpreting the regression

Outcome variable:

$$
\text{probability of donating}
$$

Meaning:

- \(1 =\) donor gave  
- \(0 =\) donor did not give  

So coefficients represent:**changes in donation probability.**

---

###  Intercept

Coefficient:

$$
0.0221
$$

This is the average donation probability for the omitted/base group:**donors asked to give exactly their previous highes contribution.**

So donation probability in this base group was about:

$$
2.21\%
$$

---

### 1.25× suggested amount coefficient

Coefficient:

$$
0.0001
$$

Meaning: **Asking for \(1.25\times\) the previous highest donation increased donation probability by only about 0.01 percentage points relative to asking for the same previous amount.**

Approximate donation probability:

$$
2.21\% + 0.01\% \approx 2.22\%
$$

Very tiny effect.

And:

$$
p\text{-value} = 0.958
$$

which is extremely large. So this effect is NOT statistically significant.

---

### 1.5× suggested amount coefficient

Coefficient:

$$
0.0009
$$

Meaning:**Asking for \(1.5\times\) the previous highest donation increased donation probability by about 0.09 percentage points relative to the base group.**

Approximate donation probability:

$$
2.21\% + 0.09\% \approx 2.30\%
$$

But:

$$
p\text{-value} = 0.662
$$

Again much larger than:

$$
0.05
$$

So this effect is also NOT statistically significant.

---

### Confidence intervals

Both confidence intervals cross:

$$
0
$$

So evidence is weak.

---

As a conclusion:

There is no strong evidence that different suggested ask amounts affected donation probability.

their previous highest contribution produced only tiny and statistically insignificant differences relative to asking for their previous highest contribution itself.

# For the following code 

We will Including following Covariates to see what happen

    Already donated in 2005
    Female
    Couple
    Number of months since last donation
    Highest previous contribution
    Number of prior donations
    Number of years since initial donation 



In [20]:


# Regression without covariates
mod_no_cov = smf.ols(
    formula="out_gavedum ~ treatment",
    data=df
)

res_no_cov = mod_no_cov.fit(cov_type="HC3")

print("Regression without covariates")
print(res_no_cov.summary())


# Regression with covariates
mod_cov = smf.ols(
    formula="""
    out_gavedum ~ treatment 
    + year5 
    + female 
    + couple 
    + mrm2 
    + hpa 
    + freq 
    + years
    """,
    data=df
)

res_cov = mod_cov.fit(cov_type="HC3")

print("\nRegression with covariates")
print(res_cov.summary())

Regression without covariates
                            OLS Regression Results                            
Dep. Variable:            out_gavedum   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     12.90
Date:                Thu, 14 May 2026   Prob (F-statistic):           0.000329
Time:                        14:46:31   Log-Likelihood:                 24504.
No. Observations:               46513   AIC:                        -4.900e+04
Df Residuals:                   46511   BIC:                        -4.899e+04
Df Model:                           1                                         
Covariance Type:                  HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.0176 

In [21]:
print("Treatment effect without covariates:", res_no_cov.params["treatment"])
print("Treatment effect with covariates:", res_cov.params["treatment"])

print("P-value without covariates:", res_no_cov.pvalues["treatment"])
print("P-value with covariates:", res_cov.pvalues["treatment"])

Treatment effect without covariates: 0.004852236752017132
Treatment effect with covariates: 0.004761193253399905
P-value without covariates: 0.00032888040540354324
P-value with covariates: 0.0003761781818938433



For this question we kept

$$
\text{treatment}
$$

and use the full dataset, not only the treated subset. Because this regression is estimating the: **overall Average Treatment Effect (ATE)**  and **matching offer vs no matching offer.**

---

The key explanatory variable is:

$$
\text{treatment}
$$

which indicates whether the donor received a matching offer or no matching offer  

---

The additional variables:

- year5  
- female  
- couple  
- mrm2  
- hpa  
- freq  
- years  

are included as **control variables (covariates).** These variables help account for differences in donor characteristics.

---

This regression is basically asking: **After accounting for donor characteristics, does the matching offer still affect donation probability?**


The treatment coefficient now measures:**the treatment effect holding donor characteristics constant.**

So the regression separates  the effect of the matching offer from differences caused by donor characteristics.

---

#### As a quick short note using only the treated dataset would remove the comparison group. Then we could no longer estimate:

$$
\text{matching offer vs no matching offer}
$$

because everyone would already be treated. That is why we must use **the full dataset.**



### Interpertation of the result


Including covariates did not meaningfully change the estimated treatment effect.

Without covariates, the estimated treatment effect of receiving a matching offer was approximately 0.00485, meaning the matching campaign increased the probability of donating by about 0.485 percentage points. The effect was highly statistically significant with a p-value of about 0.00033.

After adding donor characteristics and donation history variables (year5, female, couple, mrm2, hpa, freq, and years), the treatment effect remained extremely similar at approximately 0.00476, with a p-value of about 0.00038. The effect therefore remained positive and statistically significant at the 95% confidence level.

The fact that the treatment coefficient changed only slightly after controlling for covariates suggests that the **original randomization worked well.** In other words, treatment and control groups were already balanced before treatment, so adding controls did not substantially alter the estimated causal effect. **This strengthens confidence that the observed increase in donation probability was caused by the matching offer itself rather than by differences in donor characteristics.**

The regression with covariates also reveals some relationships between donor history and donation behavior. Donors who donated more frequently in the past (freq) were significantly more likely to donate again, while donors whose last donation was further in the past (mrm2) were less likely to donate. Similarly, donors with more years since their initial donation (years) were slightly less likely to respond. Variables such as being female, being part of a couple, donating in 2005, and highest previous contribution (hpa) did not have statistically significant effects in this specification.

Finally, the model’s explanatory power increased after adding covariates. The R-squared rose from essentially 0.000 to about 0.019, meaning the donor characteristics explain some additional variation in donation behavior, although the overall donation decision remains difficult to predict because donations are relatively rare events in the dataset.

# For the following code :

we will test correlation by create a dataframe with a subset of columns, including the outcome variable and the 7 covariates listed above.

In [23]:
# create smaller dataframe
corr_df = df[[
    'out_gavedum',
    'year5',
    'female',
    'couple',
    'mrm2',
    'hpa',
    'freq',
    'years'
]]

corr_matrix = corr_df.corr()

corr_matrix

,out_gavedum,year5,female,couple,mrm2,hpa,freq,years
out_gavedum,1.000000,0.015355,0.005214,0.000289,-0.070025,0.011068,0.112431,0.021111
year5,0.015355,1.000000,-0.043266,0.064989,0.054198,0.205799,0.486505,0.758127
female,0.005214,-0.043266,1.000000,-0.198173,-0.013517,-0.014613,0.015323,0.003122
couple,0.000289,0.064989,-0.198173,1.000000,-0.028001,0.038112,0.060945,0.078916
mrm2,-0.070025,0.054198,-0.013517,-0.028001,1.000000,0.007017,-0.116325,0.083117
hpa,0.011068,0.205799,-0.014613,0.038112,0.007017,1.000000,0.201573,0.230623
freq,0.112431,0.486505,0.015323,0.060945,-0.116325,0.201573,1.000000,0.643789
years,0.021111,0.758127,0.003122,0.078916,0.083117,0.230623,0.643789,1.000000



## Corrolation matrix interpertation 

The correlation matrix shows that most variables have only weak correlations with the outcome variable (out_gavedum).
which measures whether the donor gave anything. The strongest relationship with donation behavior is: freq = 0.112

This is a positive correlation, meaning donors who donated more frequently in the past were more likely to donate again in this campaign. This matches the earlier regression result where freq was statistically significant and positive.

Another noticeable relationship is: mrm2 = -0.070

which is negative. This means donors whose last donation was further in the past were less likely to donate now. This also matches the earlier regression where mrm2 had a statistically significant negative coefficient.

The remaining variables have very weak correlations with donation behavior. 

This makes intuitive sense because donors who have been donating for many years are also more likely to have donated recently and more frequently.

# For the following code 

We will calculate heterogeneous treatment effects for red-states by subsetting the data 

    -  The Average treatment effect for red states & confidence interval
    -  The Average treatment effect for blue states and the confidence interval on the treatment effects



In [26]:
# For red states

df_red = df[df['red0'] == 1]

mod_red = smf.ols(
    formula='out_gavedum ~ treatment',
    data=df_red
)

res_red = mod_red.fit(cov_type='HC3')

print("Red States Regression")
print(res_red.summary())

print("\n95% CI for Red States:")
print(res_red.conf_int())

Red States Regression
                            OLS Regression Results                            
Dep. Variable:            out_gavedum   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     23.47
Date:                Thu, 14 May 2026   Prob (F-statistic):           1.28e-06
Time:                        15:19:32   Log-Likelihood:                 9698.4
No. Observations:               18611   AIC:                        -1.939e+04
Df Residuals:                   18609   BIC:                        -1.938e+04
Df Model:                           1                                         
Covariance Type:                  HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.0144      0.0

In [27]:
# Blue states 

df_blue = df[df['red0'] == 0]

mod_blue = smf.ols(
    formula='out_gavedum ~ treatment',
    data=df_blue
)

res_blue = mod_blue.fit(cov_type='HC3')

print("\nBlue States Regression")
print(res_blue.summary())

print("\n95% CI for Blue States:")
print(res_blue.conf_int())


Blue States Regression
                            OLS Regression Results                            
Dep. Variable:            out_gavedum   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                    0.6897
Date:                Thu, 14 May 2026   Prob (F-statistic):              0.406
Time:                        15:20:18   Log-Likelihood:                 14811.
No. Observations:               27902   AIC:                        -2.962e+04
Df Residuals:                   27900   BIC:                        -2.960e+04
Df Model:                           1                                         
Covariance Type:                  HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.0197      0

## Heterogenous Treatment effect 

The heterogeneous treatment effect analysis suggests that the matching offer worked differently in red states and blue states.

For red states, the estimated treatment effect was approximately 0.0100, meaning the matching offer increased the probability of donating by about 1.0 percentage point. The control-group donation probability in red states was about 1.44%, while the treatment group donation probability increased to roughly 2.44%. The effect was highly statistically significant with a p-value smaller than 0.001. The 95% confidence interval ranged from about 0.59 to 1.40 percentage points, and because the interval does not include zero, there is strong evidence of a positive treatment effect in red states.

For blue states, the estimated treatment effect was much smaller, approximately 0.0015, meaning the matching offer increased donation probability by only about 0.15 percentage points. The p-value was 0.406, which is much larger than 0.05, so the effect is not statistically significant. The 95% confidence interval ranged from about -0.20 to 0.50 percentage points, and because this interval crosses zero, the true effect could be positive, negative, or zero. Therefore, there is no strong evidence that the matching offer affected donation behavior in blue states.

Overall, the results suggest substantial heterogeneity in treatment effects across political environments. The matching campaign appears to have been considerably more effective in red states than in blue states.

# For the following code 

We will calculate Heterogenous treatment effects with interaction terms


In [28]:


# interaction regression
mod_interaction = smf.ols(
    formula='out_gavedum ~ treatment + red0 + treatment:red0',
    data=df
)

res_interaction = mod_interaction.fit(cov_type='HC3')

print(res_interaction.summary())

print("\n95% Confidence Intervals:")
print(res_interaction.conf_int())

                            OLS Regression Results                            
Dep. Variable:            out_gavedum   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     8.159
Date:                Thu, 14 May 2026   Prob (F-statistic):           1.99e-05
Time:                        15:32:27   Log-Likelihood:                 24509.
No. Observations:               46513   AIC:                        -4.901e+04
Df Residuals:                   46509   BIC:                        -4.897e+04
Df Model:                           3                                         
Covariance Type:                  HC3                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept          0.0197      0.001     13.

## Interaction regression

Regression equation:

$$
Y_i
=
\beta_0
+
\beta_1 \text{Treatment}_i
+
\beta_2 \text{RedState}_i
+
\beta_3 (\text{Treatment}_i \times \text{RedState}_i)
+
\varepsilon_i
$$

---


### treatment

Effect of treatment in blue states (base group)


### red state indicator

Difference between: red states and blue states in the control group.


### treatment × red state interaction

Additional treatment effect in red states relative to blue states. This interaction coefficient is the key one.

If:

$$
p\text{-value} < 0.05
$$

for the interaction term, then **the treatment effect is statistically significantly different between red and blue states.**

---

### interpretation

This interaction regression confirms that the treatment effect in red states is statistically significantly different from the treatment effect in blue states.

The omitted/base group in this regression is blue-state control group.

So each coefficient is interpreted relative to that group.

---

# Intercept

Coefficient:

$$
0.0197
$$

This is the donation probability for **the control group in blue states.**

Meaning About \(1.97\%\) of control-group donors in blue states donated.

---

### Treatment coefficient

Coefficient:

$$
0.0015
$$

This is the treatment effect in blue states meaning In blue states, the matching offer increased donation probability by only about 0.15 percentage points.

But:

$$
p\text{-value} = 0.406
$$

So this effect is NOT statistically significant. The confidence interval also crosses:

$$
0
$$

Example:

$$
[-0.0020,\ 0.0050]
$$



---

### Red-state indicator coefficient

Coefficient:

$$
-0.0053
$$

This compares **red-state control donors** to **blue-state control donors.**

Meaning In the absence of treatment, donors in red states donated about 0.53 percentage points less often than donors in blue states.

This effect is statistically significant:

$$
p\text{-value} = 0.012
$$

---

### Interaction coefficient

This is the most important coefficient. Coefficient is:

$$
0.0085
$$

Meaning: The treatment effect in red states was about 0.85 percentage points larger than the treatment effect in blue states.

This interaction effect is statistically significant:

$$
p\text{-value} = 0.002
$$

and its confidence interval:

$$
[0.0032,\ 0.0138]
$$

does NOT cross:

$$
0
$$

So there is strong evidence that **the treatment effect differs across political environments.**

---

### red-state treatment effect

Blue-state treatment effect:

$$
0.0015
$$

Additional red-state effect:

$$
0.0085
$$

Total red-state treatment effect:

$$
0.0015 + 0.0085 = 0.0100
$$

which matches the earlier separate red-state regression result.

---

### conclusion

The matching offer had little effect in blue states, but had a much stronger and statistically significant positive effect in red states.

The interaction term confirms that: **this difference between red and blue states is itself statistically significant.**